## 1、导入依赖


In [1]:
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers.legacy import SGD

from art import config
from art.utils import load_dataset, get_file
from art.estimators.classification import KerasClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import h5py

print("TensorFlow版本:", tf.__version__)
print("GPU是否可用:", tf.config.list_physical_devices('GPU'))

TensorFlow版本: 2.10.0
GPU是否可用: []


In [2]:
print("Eager模式是否开启:", tf.executing_eagerly())

Eager模式是否开启: True


## 2、加载数据

In [3]:
(x_train, y_train), (x_test, y_test), min_, max_ = load_dataset('mnist')

## 3、加载普通分类器最为基准

In [ ]:
from tensorflow.keras.models import load_model

path = get_file('mnist_cnn_original.h5', extract=False, path=config.ART_DATA_PATH,
                url='https://www.dropbox.com/s/p2nyzne9chcerid/mnist_cnn_original.h5?dl=1')
classifier_model = load_model(path)
classifier = KerasClassifier(clip_values=(min_, max_), model=classifier_model, use_logits=False)

x_test_pred = np.argmax(classifier.predict(x_test), axis=1)
nb_correct_pred = np.sum(x_test_pred == np.argmax(y_test, axis=1))
print("=== 普通分类器 - 原始数据 ===")
print("正确分类: {}".format(nb_correct_pred))
print("准确率: {:.2f}%".format(nb_correct_pred / len(x_test) * 100))

## 4、构建TRADES模型

In [4]:
def build_model():
    model = Sequential()
    model.add(Conv2D(filters=32, kernel_size=(3, 3), strides=1,
                     activation="relu", input_shape=(28, 28, 1)))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=1,
                     activation="relu"))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(1024, activation="relu"))
    model.add(Dense(10, activation="softmax"))
    return model

trades_model = build_model()
trades_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 13, 13, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 5, 5, 64)         0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 1600)              0         
                                                                 
 dense (Dense)               (None, 1024)              1

## 5、 定义TRADES参数

In [5]:
beta  = 1.0
# 与上传代码一致的参数
epsilon   = 0.3
step_size = 0.01
num_steps = 40

optimizer = SGD(learning_rate=0.01, momentum=0.9)

print("TRADES参数:")
print("beta =", beta, "")
print("epsilon =", epsilon)
print("step_size =", step_size)
print("num_steps =", num_steps)
print("optimizer = SGD(lr=0.01, momentum=0.9)")

TRADES参数:
beta = 1.0 
epsilon = 0.3
step_size = 0.01
num_steps = 40
optimizer = SGD(lr=0.01, momentum=0.9)


## 6、学习率衰减

In [6]:
def adjust_learning_rate(optimizer, epoch):
    """
    论文官方学习率衰减策略：
    epoch < 55  : lr = 0.01
    epoch >= 55 : lr = 0.001
    epoch >= 75 : lr = 0.0001
    epoch >= 90 : lr = 0.00001
    """
    if epoch >= 90:
        new_lr = 0.00001
    elif epoch >= 75:
        new_lr = 0.0001
    elif epoch >= 55:
        new_lr = 0.001
    else:
        new_lr = 0.01

    optimizer.learning_rate.assign(new_lr)
    return new_lr

print("学习率衰减函数定义完成")
print("epoch 0~54 : lr = 0.01")
print("epoch 55~74 : lr = 0.001")
print("epoch 75~89 : lr = 0.0001")
print("epoch 90~99 : lr = 0.00001")

学习率衰减函数定义完成
epoch 0~54 : lr = 0.01
epoch 55~74 : lr = 0.001
epoch 75~89 : lr = 0.0001
epoch 90~99 : lr = 0.00001


## 7、定义TRADES对抗样本生成和训练函数

In [7]:
def generate_adv_trades(model, x_batch_np, epsilon, step_size, num_steps):
    """
    用纯numpy生成TRADES对抗样本
    目标：最大化 KL(f(x_natural) || f(x_adv))
    """
    # 获取自然样本预测
    pred_natural = model(x_batch_np, training=False).numpy()

    # 随机初始化扰动起点
    x_adv = x_batch_np + np.random.uniform(
        -epsilon, epsilon, x_batch_np.shape
    ).astype(np.float32)
    x_adv = np.clip(x_adv, min_, max_)

    for _ in range(num_steps):
        x_adv_tensor = tf.constant(x_adv, dtype=tf.float32)

        with tf.GradientTape() as tape:
            tape.watch(x_adv_tensor)
            pred_adv = model(x_adv_tensor, training=False)

            # 手动计算KL散度：p * log(p/q)
            pred_natural_tensor = tf.constant(pred_natural, dtype=tf.float32)
            kl = tf.reduce_mean(
                tf.reduce_sum(
                    pred_natural_tensor * tf.math.log(
                        pred_natural_tensor / (pred_adv + 1e-8) + 1e-8
                    ), axis=1
                )
            )

        grad = tape.gradient(kl, x_adv_tensor).numpy()

        # 沿梯度方向最大化KL散度
        x_adv = x_adv + step_size * np.sign(grad)

        # 投影回epsilon球内
        x_adv = np.clip(x_adv, x_batch_np - epsilon, x_batch_np + epsilon)
        x_adv = np.clip(x_adv, min_, max_)

    return x_adv.astype(np.float32)


def train_step_trades(model, x_batch_np, y_batch_np):
    """
    TRADES单步训练
    Loss = CE(f(x), y) + (1/beta) * KL(f(x) || f(x_adv))
    """
    # 第一步：生成对抗样本
    x_adv_np = generate_adv_trades(model, x_batch_np, epsilon, step_size, num_steps)

    x_natural_tensor = tf.constant(x_batch_np, dtype=tf.float32)
    x_adv_tensor     = tf.constant(x_adv_np,   dtype=tf.float32)
    y_tensor         = tf.constant(y_batch_np,  dtype=tf.float32)

    # 第二步：计算TRADES损失并更新模型
    with tf.GradientTape() as tape:
        pred_natural = model(x_natural_tensor, training=True)
        pred_adv     = model(x_adv_tensor,     training=True)

        # 自然损失：交叉熵
        loss_natural = tf.keras.losses.CategoricalCrossentropy()(
            y_tensor, pred_natural
        )

        # 鲁棒损失：KL散度（手动计算，数值更稳定）
        pred_natural_stop = tf.stop_gradient(pred_natural)
        kl_robust = tf.reduce_mean(
            tf.reduce_sum(
                pred_natural_stop * tf.math.log(
                    pred_natural_stop / (pred_adv + 1e-8) + 1e-8
                ), axis=1
            )
        )

        # TRADES总损失
        loss = loss_natural + (1.0 / beta) * kl_robust

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    # 计算自然准确率
    acc = tf.reduce_mean(
        tf.cast(
            tf.equal(tf.argmax(pred_natural, axis=1),
                     tf.argmax(y_tensor, axis=1)),
            tf.float32
        )
    ).numpy()

    return loss.numpy(), acc

print("TRADES训练函数定义完成")

TRADES训练函数定义完成


## 8、TRADES训练主循环（100轮+学习率衰减）

In [8]:
nb_epochs  = 1
batch_size = 128
nb_batches = len(x_train) // batch_size

print("开始 TRADES 对抗训练（论文官方设置）...")
print("Epochs: {}, Batch size: {}, 训练集大小: {}".format(
    nb_epochs, batch_size, len(x_train)))
print("=" * 55)

for epoch in range(nb_epochs):

    # 每轮开始前调整学习率
    current_lr = adjust_learning_rate(optimizer, epoch)

    # 打乱数据
    idx = np.random.permutation(len(x_train))
    x_shuffled = x_train[idx]
    y_shuffled = y_train[idx]

    epoch_loss = 0.0
    epoch_acc  = 0.0

    for batch_idx in range(nb_batches):
        start = batch_idx * batch_size
        end  = start + batch_size

        x_batch_np = x_shuffled[start:end]
        y_batch_np = y_shuffled[start:end]

        loss, acc = train_step_trades(trades_model, x_batch_np, y_batch_np)
        epoch_loss += loss
        epoch_acc  += acc

        # 每100个batch打印一次进度（与上传代码log_interval=100一致）
        if (batch_idx + 1) % 100 == 0:
            print("  Epoch [{}/{}] Batch [{}/{}]  Loss: {:.6f}  Acc: {:.0f}%  lr: {}".format(
                epoch + 1, nb_epochs,
                batch_idx + 1, nb_batches,
                loss, acc * 100, current_lr))

    avg_loss = epoch_loss / nb_batches
    avg_acc  = epoch_acc  / nb_batches
    print("Epoch [{}/{}] 完成  lr: {}  平均Loss: {:.4f}  平均Acc: {:.2f}%".format(
        epoch + 1, nb_epochs, current_lr, avg_loss, avg_acc * 100))
    print("-" * 55)

print("TRADES 训练完成！")

开始 TRADES 对抗训练（论文官方设置）...
Epochs: 1, Batch size: 128, 训练集大小: 60000
  Epoch [1/1] Batch [100/468]  Loss: 1.402348  Acc: 87%  lr: 0.01
  Epoch [1/1] Batch [200/468]  Loss: 1.083246  Acc: 94%  lr: 0.01
  Epoch [1/1] Batch [300/468]  Loss: 0.868881  Acc: 96%  lr: 0.01
  Epoch [1/1] Batch [400/468]  Loss: 0.633750  Acc: 96%  lr: 0.01
Epoch [1/1] 完成  lr: 0.01  平均Loss: 1.1204  平均Acc: 88.30%
-------------------------------------------------------
TRADES 训练完成！


In [11]:
trades_model.save('./trades_mnist_model.h5')
print("模型已保存至 ./trades_mnist_model.h5")

模型已保存至 ./trades_mnist_model.h5
